# 03 · Model selection (Team 10, Saideep)

**Deliverable 4.** Score several justified candidates on the Deliverable 3 feature set
(`features.py`) with **one** validation scheme and **one** set of metrics, then pick the
baseline model to carry into Deliverable 5.

| candidate | why it is in the comparison |
|---|---|
| Cabin median | Floor. Any model has to beat "charge the cabin's typical price". |
| Match-the-competitor rule | The obvious business heuristic: competitor price × the typical AA/competitor ratio. EDA showed a 0.99 correlation, so this is the baseline that actually matters. |
| Ridge on log price | Readable, regularised linear model; D3 made the effects roughly additive on the log scale. |
| Ridge on log(price / competitor) | Same model, but it predicts the AA/competitor ratio, so the competitor rule is its starting point and Ridge only learns adjustments. Tests whether the other features add anything *on top of* the competitor quote. |
| Random Forest | Bagged trees: non-linear, low tuning burden, robust on small data. |
| XGBoost | Gradient-boosted trees: the family D3 pointed to for the 14-day cliff and cabin × timing interactions. |

**Excluded, with reasons:**

- **ARIMA**: needs an ordered series. This file has no search or departure date, so there is
  nothing to index a time series by. Disqualified by data shape, before any compute is spent.
- **LSTM**: same problem (no sequence). It would also need far more than 199 rows, and
  TensorFlow is not a team dependency. Disqualified.

**Validation:** repeated 5-fold CV (5 repeats, `random_state=10`), giving 25 train/test fits per
model. With 199 rows a single 80/20 split (40 test rows) is too noisy to rank models on. No time
column exists, so the folds are random. All models train on `log1p(price)`; every metric is
computed in **dollars** after back-transforming.

**Metrics:** MAE (headline: dollars off per quote), RMSE (punishes big misses), R²,
and MAPE (relative error, comparable across cabins).

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import (make_scorer, mean_absolute_error,
                             mean_absolute_percentage_error, mean_squared_error, r2_score)
from sklearn.model_selection import RepeatedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

HERE = next(p for p in [Path.cwd(), Path.cwd() / "team-10-modeling" / "notebooks"]
            if (p / "load_data.py").exists())
sys.path.insert(0, str(HERE))
RESULTS = HERE.parent / "results"
VISUALS = HERE.parent / "visuals"
VISUALS.mkdir(exist_ok=True)

from load_data import load_data
from features import FEATURES, FEATURES_NO_COMPETITOR, build_features, clean

SEED = 10
df = clean(load_data())
X, y = build_features(df)
print("X:", X.shape)

## Candidates

The two baselines are written as tiny scikit-learn estimators so they go through exactly the
same CV loop as the real models: same folds, same metrics.

In [ ]:
class CabinMedian(BaseEstimator, RegressorMixin):
    """Predict the training-set median price of the quote's cabin."""
    def fit(self, X, y):
        self.medians_ = pd.Series(np.asarray(y)).groupby(np.asarray(X["cabin_rank"])).median()
        self.overall_ = float(np.median(y))
        return self

    def predict(self, X):
        return X["cabin_rank"].map(self.medians_).fillna(self.overall_).to_numpy()


class MatchCompetitor(BaseEstimator, RegressorMixin):
    """Predict competitor price x the training-set median (AA price / competitor price)."""
    def fit(self, X, y):
        comp = np.exp(X["log_competitor_price"].to_numpy())
        self.ratio_ = float(np.median(np.asarray(y) / comp))
        return self

    def predict(self, X):
        return self.ratio_ * np.exp(X["log_competitor_price"].to_numpy())


class RidgeOnRatio(BaseEstimator, RegressorMixin):
    """Ridge on log(price) - log(competitor price); the competitor quote is a fixed offset."""
    def fit(self, X, y):
        offset = X["log_competitor_price"].to_numpy()
        self.model_ = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 25)))
        self.model_.fit(X, np.log(np.asarray(y)) - offset)
        return self

    def predict(self, X):
        return np.exp(self.model_.predict(X) + X["log_competitor_price"].to_numpy())


def log_target(model):
    return TransformedTargetRegressor(regressor=model, func=np.log1p, inverse_func=np.expm1)


def candidates():
    return {
        "Ridge (log price)": log_target(make_pipeline(
            StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 25)))),
        "Random Forest": log_target(RandomForestRegressor(
            n_estimators=300, min_samples_leaf=2, random_state=SEED, n_jobs=-1)),
        "XGBoost": log_target(XGBRegressor(
            n_estimators=400, learning_rate=0.05, max_depth=3, subsample=0.8,
            colsample_bytree=0.8, min_child_weight=2, reg_lambda=1.0,
            random_state=SEED, n_jobs=2)),
    }

## Cross-validated comparison

Every candidate is scored twice: with the full feature set, and without
`log_competitor_price` (the sensitivity check promised in D3).

In [ ]:
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=SEED)
scoring = {
    "MAE": make_scorer(mean_absolute_error, greater_is_better=False),
    "RMSE": make_scorer(lambda a, p: np.sqrt(mean_squared_error(a, p)), greater_is_better=False),
    "R2": make_scorer(r2_score),
    "MAPE": make_scorer(mean_absolute_percentage_error, greater_is_better=False),
}

runs = [("Cabin median", CabinMedian(), FEATURES),
        ("Match-the-competitor rule", MatchCompetitor(), FEATURES),
        ("Ridge on log(price / competitor)", RidgeOnRatio(), FEATURES)]
for name, model in candidates().items():
    runs.append((name, model, FEATURES))
for name, model in candidates().items():
    runs.append((name, model, FEATURES_NO_COMPETITOR))

rows, fold_mae = [], {}
for name, model, cols in runs:
    feat_label = "all" if cols == FEATURES else "no competitor"
    t0 = time.perf_counter()
    res = cross_validate(model, X[cols], y, cv=cv, scoring=scoring)
    elapsed = time.perf_counter() - t0
    row = {"model": name, "features": feat_label}
    for m in scoring:
        vals = res[f"test_{m}"] * (1 if m == "R2" else -1)
        row[f"{m}_mean"] = vals.mean()
        row[f"{m}_std"] = vals.std()
    row["fit_s_per_fold"] = res["fit_time"].mean()
    rows.append(row)
    fold_mae[(name, feat_label)] = -res["test_MAE"]
    print(f"{name:28s} [{feat_label:13s}] MAE {row['MAE_mean']:7.1f} ± {row['MAE_std']:5.1f}"
          f"   R2 {row['R2_mean']:.3f}   ({elapsed:.1f}s)")

scores = pd.DataFrame(rows)
scores.round(3)

## Is the winner's margin real?

The 25 folds are shared, so models can be compared fold by fold. A model "wins" only if it
beats the other in most folds, not just on the average.

In [ ]:
def paired(a, b):
    d = fold_mae[a] - fold_mae[b]
    return {"A": f"{a[0]} [{a[1]}]", "B": f"{b[0]} [{b[1]}]",
            "mean MAE diff (A-B)": round(d.mean(), 2),
            "folds A better": int((d < 0).sum()), "of": len(d)}

comparisons = pd.DataFrame([
    paired(("Ridge (log price)", "all"), ("Match-the-competitor rule", "all")),
    paired(("Ridge on log(price / competitor)", "all"), ("Match-the-competitor rule", "all")),
    paired(("XGBoost", "all"), ("Ridge (log price)", "all")),
    paired(("Random Forest", "all"), ("Ridge (log price)", "all")),
    paired(("XGBoost", "all"), ("Match-the-competitor rule", "all")),
    paired(("XGBoost", "no competitor"), ("Ridge (log price)", "no competitor")),
])
comparisons

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
labels = [f"{k[0]}\n[{k[1]}]" for k in fold_mae]
ax.boxplot(list(fold_mae.values()), orientation="horizontal")
ax.set_yticks(range(1, len(labels) + 1), labels, fontsize=8)
ax.set_xlabel("MAE per fold (USD) — lower is better")
ax.set_title("Repeated 5-fold CV: MAE across 25 folds")
fig.tight_layout()
fig.savefig(VISUALS / "d4_cv_mae_boxplot.png", dpi=120)
plt.show()

## Write the comparison table

`results/cv_scores.md` is regenerated on every run; `results/model-comparison.md` and
`results/baseline-decision.md` discuss these numbers.

In [ ]:
def fmt(m, s, d=1):
    return f"{m:.{d}f} ± {s:.{d}f}"

lines = ["| model | features | MAE ($) | RMSE ($) | R² | MAPE (%) | fit s/fold |",
         "|---|---|---|---|---|---|---|"]
for r in scores.itertuples():
    lines.append(f"| {r.model} | {r.features} | {fmt(r.MAE_mean, r.MAE_std)} | "
                 f"{fmt(r.RMSE_mean, r.RMSE_std)} | {fmt(r.R2_mean, r.R2_std, 3)} | "
                 f"{fmt(100 * r.MAPE_mean, 100 * r.MAPE_std)} | {r.fit_s_per_fold:.3f} |")
paired_lines = ["| A | B | mean MAE diff (A−B) | folds where A is better |", "|---|---|---|---|"]
for r in comparisons.itertuples():
    paired_lines.append(f"| {r.A} | {r.B} | {r._3} | {r._4} / {r.of} |")

(RESULTS / "cv_scores.md").write_text(
    "# CV scores — Team 10 (Deliverable 4)\n\n"
    "_Generated by `team-10-modeling/notebooks/03_model_selection_saideep.ipynb`. "
    "Do not hand-edit; re-run the notebook._\n\n"
    f"Repeated 5-fold CV (5 repeats, seed {SEED}) on {len(X)} cleaned rows; "
    "mean ± std over 25 folds; metrics in dollars.\n\n"
    + "\n".join(lines) + "\n\n## Paired fold comparison\n\n" + "\n".join(paired_lines) + "\n",
    encoding="utf-8")
print("wrote", RESULTS / "cv_scores.md")